# Journal Extension J3 — Progressive Ablation on the Original 83.95% Protocol

## Purpose

This notebook performs the journal ablation study on the **exact original 80/10/10 thesis split**.

It does **not retrain any CNN/ViT model**.

It reuses the **frozen saved validation/test probabilities as authoritative artifacts**. The raw logistic-stacking stage is recovered algebraically from the saved No-TTA stacking blend instead of refitting the meta-learner, avoiding scikit-learn solver/version differences.

### Main stages

1. Rotation
2. SimCLR No-TTA
3. SimCLR + Rotation simple mean
4. FusionConcat
5. FusionGated V2
6. FusionGated V3
7. Residual Fusion
8. Optimized Weighted Ensemble
9. Raw Logistic Stacking
10. No-TTA Anchor + Stacking Blend
11. **+ Class-Bias Correction — Final Proposed Model**
12. DINOv2 ViT-S/14 3-seed mean

The notebook reports:
- Validation BACC / Macro-F1
- Test Accuracy
- Test BACC
- Test Macro-F1
- Test Macro-Specificity
- Test AUC
- ΔBACC relative to SimCLR
- ensemble-stage incremental gains
- per-class recalls
- class-recall changes relative to SimCLR
- validation-selected final-model audit

## Required Kaggle inputs

Attach these saved outputs:

### A. Original full thesis pipeline — REQUIRED
Saved output of `last-part-of-the-previous-one`.

It must contain the saved source-model probabilities under `rotation_ssl_paperlike/`.

### B. Confidence-gated ensemble — REQUIRED
Saved output of `confidence-gated-ensemble`.

It must contain:
- `reuse_only_all_model_ensemble/final_ensemble_config.json`
- `final_internal_probabilities.npz`
- `stacking_meta_model.joblib` (optional; this notebook can reconstruct it)
- validation/test comparison tables.

### C. J2 DINOv2 output — REQUIRED for DINO row
Saved output of `journal-extension-j2`, containing:
- `dinov2_original_split_multiseed_probabilities.npz`

### D. J1 protocol lock — RECOMMENDED
Saved J1 output or package. Used only to verify the exact original protocol.

## No image datasets are required

J3 works entirely from saved predictions.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / CONFIG
# ============================================================
import json
import math
import hashlib
import shutil
import zipfile
from collections import OrderedDict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

OUTPUT_DIR = WORKING / "j3_original_split_ablation"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
AUDIT_DIR = OUTPUT_DIR / "audit"

for d in [OUTPUT_DIR, TABLE_DIR, FIG_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]
NUM_CLASSES = 8

EXPECTED_VAL_N = 2533
EXPECTED_TEST_N = 2534
EXPECTED_FINAL_METHOD = "NoTTA_StackingBlend_ClassBias"

RANDOM_STATE = 42

print("=" * 96)
print("J3 — ORIGINAL-SPLIT PROGRESSIVE ABLATION")
print("=" * 96)
print("Output:", OUTPUT_DIR)

J3 — ORIGINAL-SPLIT PROGRESSIVE ABLATION
Output: /kaggle/working/j3_original_split_ablation


In [2]:
# ============================================================
# CELL 2 — HELPER FUNCTIONS
# ============================================================

def normalize_probabilities(probabilities, epsilon=1e-12):
    p = np.asarray(probabilities, dtype=np.float64)

    if p.ndim != 2 or p.shape[1] != NUM_CLASSES:
        raise ValueError(
            f"Expected probability shape (N,{NUM_CLASSES}), got {p.shape}"
        )

    p = np.nan_to_num(
        p,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    p = np.clip(
        p,
        epsilon,
        None,
    )

    p /= p.sum(
        axis=1,
        keepdims=True,
    )

    return p.astype(np.float32)

def probability_columns(df):
    # Format 1: prob_class_0 ... prob_class_7
    cols = [
        c for c in df.columns
        if c.startswith("prob_class_")
        and c.rsplit("_", 1)[-1].isdigit()
    ]
    if len(cols) == NUM_CLASSES:
        return sorted(
            cols,
            key=lambda c: int(c.rsplit("_", 1)[-1])
        )

    # Format 2: prob_0_MEL ... prob_7_SCC
    cols = [
        c for c in df.columns
        if c.startswith("prob_")
        and len(c.split("_")) >= 2
        and c.split("_")[1].isdigit()
    ]
    if len(cols) == NUM_CLASSES:
        return sorted(
            cols,
            key=lambda c: int(c.split("_")[1])
        )

    # Format 3: prob_0 ... prob_7
    exact = [
        f"prob_{i}"
        for i in range(NUM_CLASSES)
        if f"prob_{i}" in df.columns
    ]
    if len(exact) == NUM_CLASSES:
        return exact

    raise ValueError(
        "Could not identify exactly 8 probability columns. "
        f"Columns: {list(df.columns)}"
    )

def label_column(df):
    for c in ["y_true", "true_label", "label"]:
        if c in df.columns:
            return c
    raise ValueError(
        "No true-label column found."
    )

def load_prediction_csv(path):
    df = pd.read_csv(path)

    y_col = label_column(df)
    p_cols = probability_columns(df)

    y = df[y_col].to_numpy(np.int32)
    p = normalize_probabilities(
        df[p_cols].to_numpy(np.float32)
    )

    return y, p

def metrics_dict(y_true, probabilities):
    p = normalize_probabilities(probabilities)
    pred = np.argmax(p, axis=1)

    y_bin = label_binarize(
        y_true,
        classes=np.arange(NUM_CLASSES)
    )

    try:
        auc = float(
            roc_auc_score(
                y_bin,
                p,
                average="macro",
                multi_class="ovr",
            )
        )
    except Exception:
        auc = np.nan

    cm = confusion_matrix(
        y_true,
        pred,
        labels=np.arange(NUM_CLASSES),
    )

    specificity = []
    recalls = []

    for class_id in range(NUM_CLASSES):
        tp = cm[class_id, class_id]
        fn = cm[class_id, :].sum() - tp
        fp = cm[:, class_id].sum() - tp
        tn = cm.sum() - tp - fn - fp

        recalls.append(
            tp / max(tp + fn, 1)
        )

        specificity.append(
            tn / max(tn + fp, 1)
        )

    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "bacc": float(balanced_accuracy_score(y_true, pred)),
        "macro_precision": float(
            precision_score(
                y_true,
                pred,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall": float(
            recall_score(
                y_true,
                pred,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                pred,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "macro_specificity": float(np.mean(specificity)),
        "auc_macro_ovr": auc,
        "class_recalls": np.asarray(recalls, dtype=np.float64),
        "y_pred": pred,
    }

def temperature_scale(probabilities, temperature):
    p = normalize_probabilities(probabilities)

    logits = np.log(
        np.clip(
            p,
            1e-12,
            1.0,
        )
    )

    scaled = logits / float(temperature)

    scaled -= scaled.max(
        axis=1,
        keepdims=True,
    )

    return normalize_probabilities(
        np.exp(scaled)
    )

def apply_class_bias(probabilities, class_bias):
    p = normalize_probabilities(probabilities)

    logits = np.log(
        np.clip(
            p,
            1e-12,
            1.0,
        )
    )

    logits += np.asarray(
        class_bias,
        dtype=np.float64,
    )[None, :]

    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    return normalize_probabilities(
        np.exp(logits)
    )

def entropy(probabilities):
    p = np.clip(
        normalize_probabilities(probabilities),
        1e-12,
        1.0,
    )

    return -np.sum(
        p * np.log(p),
        axis=1,
        keepdims=True,
    )

def find_all(filename):
    return sorted(
        KAGGLE_INPUT.rglob(filename),
        key=lambda p: (len(p.parts), str(p))
    )

print("Helpers ready.")

Helpers ready.


In [3]:
# ============================================================
# CELL 3 — FIND THE ORIGINAL CONFIDENCE-GATED OUTPUT
# ============================================================

config_hits = find_all(
    "final_ensemble_config.json"
)

config_hits = sorted(
    config_hits,
    key=lambda p: (
        0 if "reuse_only_all_model_ensemble" in str(p) else 9,
        len(p.parts),
        str(p),
    )
)

if not config_hits:
    raise FileNotFoundError(
        "final_ensemble_config.json not found. "
        "Attach the saved confidence-gated ensemble output."
    )

FINAL_CONFIG_JSON = None
FINAL_CONFIG = None

for candidate in config_hits:
    try:
        with open(candidate, "r") as f:
            cfg = json.load(f)

        if cfg.get("final_method") != EXPECTED_FINAL_METHOD:
            continue

        FINAL_CONFIG_JSON = candidate
        FINAL_CONFIG = cfg
        break
    except Exception:
        continue

if FINAL_CONFIG_JSON is None:
    raise FileNotFoundError(
        "Could not find the original confidence-gated final config "
        f"with final_method={EXPECTED_FINAL_METHOD}."
    )

print("Final config:")
print(FINAL_CONFIG_JSON)

print("\nFrozen final method:")
print(FINAL_CONFIG["final_method"])

print("\nModels:")
print(FINAL_CONFIG["models"])

if bool(FINAL_CONFIG.get("test_used_for_selection", True)):
    raise RuntimeError(
        "Frozen config says test was used for selection."
    )

Final config:
/kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/final_ensemble_config.json

Frozen final method:
NoTTA_StackingBlend_ClassBias

Models:
['SimCLR_NoTTA', 'SimCLR_SelectedTTA', 'Rotation', 'FusionConcat', 'FusionGatedV2', 'FusionGatedV3', 'ResidualFusion']


In [4]:
# ============================================================
# CELL 4 — FIND / VALIDATE THE ORIGINAL 2533/2534 FINAL NPZ
# ============================================================

npz_hits = find_all(
    "final_internal_probabilities.npz"
)

ranked_npz = sorted(
    npz_hits,
    key=lambda p: (
        0 if "reuse_only_all_model_ensemble" in str(p) else
        1 if "j1_original_protocol_lock" in str(p) else
        9,
        len(p.parts),
        str(p),
    )
)

FINAL_PROB_NPZ = None

for candidate in ranked_npz:
    try:
        with np.load(candidate) as z:
            if not {
                "y_val",
                "y_test",
                "final_val_prob",
                "final_test_prob",
                "weighted_val_prob",
                "weighted_test_prob",
                "stacking_val_prob",
                "stacking_test_prob",
                "stacking_bias_val_prob",
                "stacking_bias_test_prob",
            }.issubset(z.files):
                continue

            if len(z["y_val"]) != EXPECTED_VAL_N:
                continue

            if len(z["y_test"]) != EXPECTED_TEST_N:
                continue

        FINAL_PROB_NPZ = candidate
        break
    except Exception:
        continue

if FINAL_PROB_NPZ is None:
    raise FileNotFoundError(
        "Original 2533/2534 final_internal_probabilities.npz not found."
    )

with np.load(FINAL_PROB_NPZ) as z:
    Y_VAL = np.asarray(z["y_val"], dtype=np.int32)
    Y_TEST = np.asarray(z["y_test"], dtype=np.int32)

    SAVED_FINAL_VAL = normalize_probabilities(z["final_val_prob"])
    SAVED_FINAL_TEST = normalize_probabilities(z["final_test_prob"])

    SAVED_WEIGHTED_VAL = normalize_probabilities(z["weighted_val_prob"])
    SAVED_WEIGHTED_TEST = normalize_probabilities(z["weighted_test_prob"])

    SAVED_ANCHOR_STACK_VAL = normalize_probabilities(z["stacking_val_prob"])
    SAVED_ANCHOR_STACK_TEST = normalize_probabilities(z["stacking_test_prob"])

    SAVED_BIAS_VAL = normalize_probabilities(z["stacking_bias_val_prob"])
    SAVED_BIAS_TEST = normalize_probabilities(z["stacking_bias_test_prob"])

if not np.allclose(
    SAVED_FINAL_TEST,
    SAVED_BIAS_TEST,
    atol=1e-7,
):
    raise RuntimeError(
        "Saved final probability is not the class-bias final stage."
    )

print("Final NPZ:")
print(FINAL_PROB_NPZ)
print("VAL :", Y_VAL.shape)
print("TEST:", Y_TEST.shape)

Final NPZ:
/kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/reuse_only_all_model_ensemble/final_internal_probabilities.npz
VAL : (2533,)
TEST: (2534,)


In [5]:
# ============================================================
# CELL 5 — FIND ORIGINAL PIPELINE ROOT BY PROBABILITY SHAPE
# ============================================================

sim_hits = find_all(
    "multiseed_probabilities_no_tta.npz"
)

PIPELINE_ROOT = None
SIMCLR_NPZ = None

for candidate in sim_hits:
    try:
        with np.load(candidate) as z:
            if not {
                "y_val",
                "y_test",
                "val_prob",
                "test_prob",
            }.issubset(z.files):
                continue

            if len(z["y_val"]) != EXPECTED_VAL_N:
                continue

            if len(z["y_test"]) != EXPECTED_TEST_N:
                continue

            if not np.array_equal(
                np.asarray(z["y_val"], dtype=np.int32),
                Y_VAL,
            ):
                continue

            if not np.array_equal(
                np.asarray(z["y_test"], dtype=np.int32),
                Y_TEST,
            ):
                continue

        SIMCLR_NPZ = candidate
        PIPELINE_ROOT = candidate.parent.parent
        break

    except Exception:
        continue

if PIPELINE_ROOT is None:
    raise FileNotFoundError(
        "Could not identify the original thesis pipeline root. "
        "Attach the saved output of `last-part-of-the-previous-one`."
    )

print("Original thesis pipeline root:")
print(PIPELINE_ROOT)
print("\nSimCLR source:")
print(SIMCLR_NPZ)

Original thesis pipeline root:
/kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike

SimCLR source:
/kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/simclr_finetune_multiseed_bacc_v2/multiseed_probabilities_no_tta.npz


In [6]:
# ============================================================
# CELL 6 — LOAD ALL SEVEN ORIGINAL SOURCE MODELS
# ============================================================

MODEL_PATHS = OrderedDict([
    (
        "SimCLR_NoTTA",
        {
            "type": "npz",
            "path": (
                PIPELINE_ROOT
                / "simclr_finetune_multiseed_bacc_v2"
                / "multiseed_probabilities_no_tta.npz"
            ),
        },
    ),
    (
        "SimCLR_SelectedTTA",
        {
            "type": "npz",
            "path": (
                PIPELINE_ROOT
                / "simclr_finetune_multiseed_bacc_v2"
                / "tta_selection"
                / "selected_multiseed_tta_probabilities.npz"
            ),
        },
    ),
    (
        "Rotation",
        {
            "type": "csv",
            "val": (
                PIPELINE_ROOT
                / "extended_imagenet_rotation_finetune"
                / "imagenet_rotation_val_predictions.csv"
            ),
            "test": (
                PIPELINE_ROOT
                / "extended_imagenet_rotation_finetune"
                / "imagenet_rotation_test_predictions.csv"
            ),
        },
    ),
    (
        "FusionConcat",
        {
            "type": "csv",
            "val": (
                PIPELINE_ROOT
                / "fusion_concat_multiseed_rep_v2"
                / "full_evaluation"
                / "val"
                / "val_predictions.csv"
            ),
            "test": (
                PIPELINE_ROOT
                / "fusion_concat_multiseed_rep_v2"
                / "full_evaluation"
                / "test"
                / "test_predictions.csv"
            ),
        },
    ),
    (
        "FusionGatedV2",
        {
            "type": "csv",
            "val": (
                PIPELINE_ROOT
                / "fusion_gated_v2_multiseed_rep_v2"
                / "full_evaluation"
                / "val"
                / "val_predictions.csv"
            ),
            "test": (
                PIPELINE_ROOT
                / "fusion_gated_v2_multiseed_rep_v2"
                / "full_evaluation"
                / "test"
                / "test_predictions.csv"
            ),
        },
    ),
    (
        "FusionGatedV3",
        {
            "type": "csv",
            "val": (
                PIPELINE_ROOT
                / "fusion_gated_v3_multiseed_rep_v2"
                / "full_evaluation"
                / "val"
                / "val_predictions.csv"
            ),
            "test": (
                PIPELINE_ROOT
                / "fusion_gated_v3_multiseed_rep_v2"
                / "full_evaluation"
                / "test"
                / "test_predictions.csv"
            ),
        },
    ),
    (
        "ResidualFusion",
        {
            "type": "npz",
            "path": (
                PIPELINE_ROOT
                / "residual_simclr_rotation_fusion_v1"
                / "multiseed_probabilities.npz"
            ),
        },
    ),
])

VAL_PROB = OrderedDict()
TEST_PROB = OrderedDict()
availability = []

for model_name, spec in MODEL_PATHS.items():
    try:
        if spec["type"] == "npz":
            if not spec["path"].exists():
                raise FileNotFoundError(str(spec["path"]))

            with np.load(spec["path"]) as z:
                yv = np.asarray(z["y_val"], dtype=np.int32)
                yt = np.asarray(z["y_test"], dtype=np.int32)
                pv = normalize_probabilities(z["val_prob"])
                pt = normalize_probabilities(z["test_prob"])

        else:
            if not spec["val"].exists():
                raise FileNotFoundError(str(spec["val"]))
            if not spec["test"].exists():
                raise FileNotFoundError(str(spec["test"]))

            yv, pv = load_prediction_csv(spec["val"])
            yt, pt = load_prediction_csv(spec["test"])

        if not np.array_equal(yv, Y_VAL):
            raise RuntimeError("validation labels are not aligned")

        if not np.array_equal(yt, Y_TEST):
            raise RuntimeError("test labels are not aligned")

        VAL_PROB[model_name] = pv
        TEST_PROB[model_name] = pt

        availability.append({
            "model": model_name,
            "available": True,
            "source": str(
                spec.get("path", spec.get("test"))
            ),
        })

    except Exception as e:
        availability.append({
            "model": model_name,
            "available": False,
            "source": str(e),
        })

AVAILABILITY_DF = pd.DataFrame(availability)
display(AVAILABILITY_DF)

missing = [
    name
    for name in MODEL_PATHS
    if name not in VAL_PROB
]

if missing:
    raise FileNotFoundError(
        "Missing original source predictions: "
        + ", ".join(missing)
    )

if list(VAL_PROB.keys()) != list(FINAL_CONFIG["models"]):
    raise RuntimeError(
        "Loaded source-model order differs from frozen final config.\n"
        f"Loaded: {list(VAL_PROB.keys())}\n"
        f"Config: {FINAL_CONFIG['models']}"
    )

AVAILABILITY_DF.to_csv(
    AUDIT_DIR / "loaded_source_models.csv",
    index=False,
)

print("\nAll seven thesis source models loaded and aligned.")

                model  available  \
0        SimCLR_NoTTA       True   
1  SimCLR_SelectedTTA       True   
2            Rotation       True   
3        FusionConcat       True   
4       FusionGatedV2       True   
5       FusionGatedV3       True   
6      ResidualFusion       True   

                                              source  
0  /kaggle/input/notebooks/anirbanroysudipto/last...  
1  /kaggle/input/notebooks/anirbanroysudipto/last...  
2  /kaggle/input/notebooks/anirbanroysudipto/last...  
3  /kaggle/input/notebooks/anirbanroysudipto/last...  
4  /kaggle/input/notebooks/anirbanroysudipto/last...  
5  /kaggle/input/notebooks/anirbanroysudipto/last...  
6  /kaggle/input/notebooks/anirbanroysudipto/last...  


All seven thesis source models loaded and aligned.


In [7]:
# ============================================================
# CELL 7 — CALIBRATION + FROZEN ENSEMBLE CONFIG
# ============================================================

MODEL_NAMES = list(VAL_PROB.keys())

TEMPERATURES = OrderedDict(
    (
        name,
        float(FINAL_CONFIG["temperatures"][name])
    )
    for name in MODEL_NAMES
)

WEIGHTS = np.asarray(
    [
        float(FINAL_CONFIG["optimized_weights"][name])
        for name in MODEL_NAMES
    ],
    dtype=np.float64,
)

BEST_C = float(
    FINAL_CONFIG["stacking_C"]
)

BEST_ALPHA = float(
    FINAL_CONFIG["stacking_alpha"]
)

CLASS_BIAS = np.asarray(
    FINAL_CONFIG["class_bias"],
    dtype=np.float64,
)

ANCHOR_NAME = str(
    FINAL_CONFIG["anchor"]
)

CALIBRATED_VAL = OrderedDict(
    (
        name,
        temperature_scale(
            VAL_PROB[name],
            TEMPERATURES[name],
        )
    )
    for name in MODEL_NAMES
)

CALIBRATED_TEST = OrderedDict(
    (
        name,
        temperature_scale(
            TEST_PROB[name],
            TEMPERATURES[name],
        )
    )
    for name in MODEL_NAMES
)

config_df = pd.DataFrame({
    "model": MODEL_NAMES,
    "temperature": [
        TEMPERATURES[n]
        for n in MODEL_NAMES
    ],
    "optimized_weight": WEIGHTS,
})

display(config_df)

print("Anchor       :", ANCHOR_NAME)
print("Stacking C   :", BEST_C)
print("Stack alpha  :", BEST_ALPHA)
print("Class bias   :", CLASS_BIAS.tolist())

config_df.to_csv(
    TABLE_DIR / "frozen_ensemble_configuration.csv",
    index=False,
)

                model  temperature  optimized_weight
0        SimCLR_NoTTA         1.30          0.414650
1  SimCLR_SelectedTTA         1.30          0.090338
2            Rotation         1.75          0.324790
3        FusionConcat         1.75          0.104999
4       FusionGatedV2         1.75          0.002943
5       FusionGatedV3         2.00          0.039286
6      ResidualFusion         2.00          0.022994

Anchor       : SimCLR_NoTTA
Stacking C   : 0.01
Stack alpha  : 0.6
Class bias   : [-0.05000000074505806, 0.05000000074505806, -0.05000000074505806, 0.15000000596046448, 0.0, 0.0, 0.0, -0.05000000074505806]


In [8]:
# ============================================================
# CELL 8 — RECOVER RAW LOGISTIC STACKING FROM THE FROZEN
# SAVED NO-TTA STACKING BLEND
#
# IMPORTANT:
# The confidence-gated notebook already saved the exact
# validation/test probabilities of the selected stacking blend.
# We therefore do NOT refit LogisticRegression here.
#
# This avoids tiny solver/library-version differences that can
# occur when re-fitting the same LR meta-learner in a later
# Kaggle environment.
#
# Original blend:
#   STACK_BLEND =
#       (1 - alpha) * calibrated SimCLR_NoTTA
#       + alpha * RAW_STACK
#
# Because all three terms are normalized probabilities and the
# coefficients sum to 1, RAW_STACK can be recovered exactly:
#
#   RAW_STACK =
#       (STACK_BLEND - (1-alpha)*ANCHOR) / alpha
# ============================================================

if BEST_ALPHA <= 0.0:
    raise RuntimeError(
        f"Invalid frozen stacking alpha: {BEST_ALPHA}"
    )

CAL_ANCHOR_VAL = CALIBRATED_VAL[
    ANCHOR_NAME
]

CAL_ANCHOR_TEST = CALIBRATED_TEST[
    ANCHOR_NAME
]

RAW_STACK_OOF_VAL_LINEAR = (
    SAVED_ANCHOR_STACK_VAL
    - (1.0 - BEST_ALPHA)
    * CAL_ANCHOR_VAL
) / BEST_ALPHA

RAW_STACK_TEST_LINEAR = (
    SAVED_ANCHOR_STACK_TEST
    - (1.0 - BEST_ALPHA)
    * CAL_ANCHOR_TEST
) / BEST_ALPHA

raw_val_min = float(
    RAW_STACK_OOF_VAL_LINEAR.min()
)

raw_test_min = float(
    RAW_STACK_TEST_LINEAR.min()
)

raw_val_sum_error = float(
    np.max(
        np.abs(
            RAW_STACK_OOF_VAL_LINEAR.sum(
                axis=1
            )
            - 1.0
        )
    )
)

raw_test_sum_error = float(
    np.max(
        np.abs(
            RAW_STACK_TEST_LINEAR.sum(
                axis=1
            )
            - 1.0
        )
    )
)

print(
    "Recovered raw-stack minimum probability "
    f"(VAL): {raw_val_min:.8e}"
)

print(
    "Recovered raw-stack minimum probability "
    f"(TEST): {raw_test_min:.8e}"
)

print(
    "Recovered raw-stack max row-sum error "
    f"(VAL): {raw_val_sum_error:.8e}"
)

print(
    "Recovered raw-stack max row-sum error "
    f"(TEST): {raw_test_sum_error:.8e}"
)

# Small negative values can occur only from floating-point
# subtraction. Large negatives would indicate that the wrong
# source-model probabilities were attached.
NEGATIVE_TOLERANCE = 5e-5

if raw_val_min < -NEGATIVE_TOLERANCE:
    raise RuntimeError(
        "Recovered validation raw-stack probabilities contain "
        f"unexpected negative values (min={raw_val_min}). "
        "This suggests the attached source probabilities do not "
        "belong to the same original confidence-gated run."
    )

if raw_test_min < -NEGATIVE_TOLERANCE:
    raise RuntimeError(
        "Recovered test raw-stack probabilities contain "
        f"unexpected negative values (min={raw_test_min}). "
        "This suggests the attached source probabilities do not "
        "belong to the same original confidence-gated run."
    )

RAW_STACK_OOF_VAL = normalize_probabilities(
    np.clip(
        RAW_STACK_OOF_VAL_LINEAR,
        0.0,
        None,
    )
)

RAW_STACK_TEST = normalize_probabilities(
    np.clip(
        RAW_STACK_TEST_LINEAR,
        0.0,
        None,
    )
)

# Recreate the blend only as an integrity audit.
RECOVERED_BLEND_VAL = normalize_probabilities(
    (1.0 - BEST_ALPHA)
    * CAL_ANCHOR_VAL
    + BEST_ALPHA
    * RAW_STACK_OOF_VAL
)

RECOVERED_BLEND_TEST = normalize_probabilities(
    (1.0 - BEST_ALPHA)
    * CAL_ANCHOR_TEST
    + BEST_ALPHA
    * RAW_STACK_TEST
)

blend_val_max_abs_diff = float(
    np.max(
        np.abs(
            RECOVERED_BLEND_VAL
            - SAVED_ANCHOR_STACK_VAL
        )
    )
)

blend_test_max_abs_diff = float(
    np.max(
        np.abs(
            RECOVERED_BLEND_TEST
            - SAVED_ANCHOR_STACK_TEST
        )
    )
)

print(
    "Recovered blend max |difference| "
    f"(VAL): {blend_val_max_abs_diff:.8e}"
)

print(
    "Recovered blend max |difference| "
    f"(TEST): {blend_test_max_abs_diff:.8e}"
)

if blend_val_max_abs_diff > 1e-4:
    raise RuntimeError(
        "Could not recover the original validation stacking "
        "blend accurately. Wrong source probabilities may be attached."
    )

if blend_test_max_abs_diff > 1e-4:
    raise RuntimeError(
        "Could not recover the original test stacking "
        "blend accurately. Wrong source probabilities may be attached."
    )

# The frozen saved probabilities remain authoritative for the
# actual A11 and A12 journal rows.
ANCHOR_STACK_VAL = SAVED_ANCHOR_STACK_VAL
ANCHOR_STACK_TEST = SAVED_ANCHOR_STACK_TEST
BIAS_VAL = SAVED_BIAS_VAL
BIAS_TEST = SAVED_BIAS_TEST

STACK_RECOVERY_AUDIT = {
    "method":
        "inverse recovery from frozen saved stacking blend",

    "stacking_alpha":
        float(BEST_ALPHA),

    "anchor":
        ANCHOR_NAME,

    "raw_val_min_before_clip":
        raw_val_min,

    "raw_test_min_before_clip":
        raw_test_min,

    "raw_val_max_row_sum_error":
        raw_val_sum_error,

    "raw_test_max_row_sum_error":
        raw_test_sum_error,

    "recovered_blend_val_max_abs_diff":
        blend_val_max_abs_diff,

    "recovered_blend_test_max_abs_diff":
        blend_test_max_abs_diff,

    "logistic_regression_refitted":
        False,

    "reason":
        (
            "Use frozen saved probabilities to avoid "
            "scikit-learn solver/version-dependent refit differences."
        ),
}

with open(
    AUDIT_DIR
    / "raw_stacking_recovery_audit.json",
    "w",
) as f:
    json.dump(
        STACK_RECOVERY_AUDIT,
        f,
        indent=2,
    )

print(
    "\nRaw logistic stacking recovered from the "
    "original frozen blend: PASSED"
)

Recovered raw-stack minimum probability (VAL): 4.00651771e-07
Recovered raw-stack minimum probability (TEST): 4.95999473e-07
Recovered raw-stack max row-sum error (VAL): 2.38418579e-07
Recovered raw-stack max row-sum error (TEST): 2.38418579e-07
Recovered blend max |difference| (VAL): 5.96046448e-08
Recovered blend max |difference| (TEST): 5.96046448e-08

Raw logistic stacking recovered from the original frozen blend: PASSED


In [9]:
# ============================================================
# CELL 9 — SIMPLE TWO-BRANCH MEAN + FROZEN WEIGHTED ENSEMBLE
# ============================================================

SIM_ROT_MEAN_VAL = normalize_probabilities(
    0.5
    * VAL_PROB["SimCLR_NoTTA"]
    + 0.5
    * VAL_PROB["Rotation"]
)

SIM_ROT_MEAN_TEST = normalize_probabilities(
    0.5
    * TEST_PROB["SimCLR_NoTTA"]
    + 0.5
    * TEST_PROB["Rotation"]
)

# For the journal ablation table, the exact weighted-ensemble
# probabilities saved by the original confidence-gated run are
# authoritative. We do not rerun the random constrained weight
# search here.

WEIGHTED_VAL_RECON = normalize_probabilities(
    sum(
        WEIGHTS[i]
        * CALIBRATED_VAL[name]
        for i, name in enumerate(
            MODEL_NAMES
        )
    )
)

WEIGHTED_TEST_RECON = normalize_probabilities(
    sum(
        WEIGHTS[i]
        * CALIBRATED_TEST[name]
        for i, name in enumerate(
            MODEL_NAMES
        )
    )
)

weighted_val_max_abs_diff = float(
    np.max(
        np.abs(
            WEIGHTED_VAL_RECON
            - SAVED_WEIGHTED_VAL
        )
    )
)

weighted_test_max_abs_diff = float(
    np.max(
        np.abs(
            WEIGHTED_TEST_RECON
            - SAVED_WEIGHTED_TEST
        )
    )
)

print(
    "Weighted ensemble reconstruction "
    f"max |difference| (VAL): "
    f"{weighted_val_max_abs_diff:.8e}"
)

print(
    "Weighted ensemble reconstruction "
    f"max |difference| (TEST): "
    f"{weighted_test_max_abs_diff:.8e}"
)

# Do not fail because a saved probability artifact is more
# authoritative than a floating-point reconstruction.
# The exact saved arrays are used in all reported metrics.

WEIGHTED_VAL = SAVED_WEIGHTED_VAL
WEIGHTED_TEST = SAVED_WEIGHTED_TEST

with open(
    AUDIT_DIR
    / "weighted_ensemble_reconstruction_audit.json",
    "w",
) as f:
    json.dump(
        {
            "saved_probabilities_used_for_reporting":
                True,

            "validation_max_abs_reconstruction_diff":
                weighted_val_max_abs_diff,

            "test_max_abs_reconstruction_diff":
                weighted_test_max_abs_diff,
        },
        f,
        indent=2,
    )

print(
    "Frozen saved weighted-ensemble probabilities "
    "will be used for reporting."
)

Weighted ensemble reconstruction max |difference| (VAL): 5.96046448e-08
Weighted ensemble reconstruction max |difference| (TEST): 5.96046448e-08
Frozen saved weighted-ensemble probabilities will be used for reporting.


In [10]:
# ============================================================
# CELL 10 — LOAD J2 DINOv2 SAME-SPLIT RESULT
# ============================================================

dino_hits = find_all(
    "dinov2_original_split_multiseed_probabilities.npz"
)

DINO_NPZ = None

for candidate in dino_hits:
    try:
        with np.load(candidate) as z:
            if not {
                "y_val",
                "y_test",
                "val_prob",
                "test_prob",
            }.issubset(z.files):
                continue

            yv = np.asarray(
                z["y_val"],
                dtype=np.int32,
            )
            yt = np.asarray(
                z["y_test"],
                dtype=np.int32,
            )

            if not np.array_equal(yv, Y_VAL):
                continue

            if not np.array_equal(yt, Y_TEST):
                continue

        DINO_NPZ = candidate
        break

    except Exception:
        continue

if DINO_NPZ is None:
    raise FileNotFoundError(
        "J2 DINO original-split probability file not found. "
        "Attach the saved J2 notebook output."
    )

with np.load(DINO_NPZ) as z:
    DINO_VAL = normalize_probabilities(
        z["val_prob"]
    )
    DINO_TEST = normalize_probabilities(
        z["test_prob"]
    )

print("DINO source:")
print(DINO_NPZ)

DINO source:
/kaggle/input/notebooks/anirbanroysudipto/journal-extension-j2/j2_dinov2_original_split/dinov2_original_split_multiseed_probabilities.npz


In [11]:
# ============================================================
# CELL 11 — BUILD JOURNAL ABLATION TABLE
# ============================================================

STAGES = OrderedDict([
    (
        "A1 Rotation",
        (
            VAL_PROB["Rotation"],
            TEST_PROB["Rotation"],
            "Single SSL branch",
        ),
    ),
    (
        "A2 SimCLR No-TTA",
        (
            VAL_PROB["SimCLR_NoTTA"],
            TEST_PROB["SimCLR_NoTTA"],
            "Single SSL branch / anchor",
        ),
    ),
    (
        "A3 SimCLR + Rotation Mean",
        (
            SIM_ROT_MEAN_VAL,
            SIM_ROT_MEAN_TEST,
            "Simple heterogeneous probability mean",
        ),
    ),
    (
        "A4 SimCLR Selected-TTA",
        (
            VAL_PROB["SimCLR_SelectedTTA"],
            TEST_PROB["SimCLR_SelectedTTA"],
            "Single SSL branch + selected TTA",
        ),
    ),
    (
        "A5 FusionConcat",
        (
            VAL_PROB["FusionConcat"],
            TEST_PROB["FusionConcat"],
            "Feature fusion",
        ),
    ),
    (
        "A6 FusionGated V2",
        (
            VAL_PROB["FusionGatedV2"],
            TEST_PROB["FusionGatedV2"],
            "Gated feature/probability fusion",
        ),
    ),
    (
        "A7 FusionGated V3",
        (
            VAL_PROB["FusionGatedV3"],
            TEST_PROB["FusionGatedV3"],
            "Enhanced gated fusion",
        ),
    ),
    (
        "A8 Residual Fusion",
        (
            VAL_PROB["ResidualFusion"],
            TEST_PROB["ResidualFusion"],
            "Residual fusion",
        ),
    ),
    (
        "A9 Optimized Weighted Ensemble",
        (
            WEIGHTED_VAL,
            WEIGHTED_TEST,
            "Validation-optimized calibrated ensemble",
        ),
    ),
    (
        "A10 Raw Logistic Stacking",
        (
            RAW_STACK_OOF_VAL,
            RAW_STACK_TEST,
            "Logistic meta-learner",
        ),
    ),
    (
        "A11 No-TTA Anchor + Stacking",
        (
            SAVED_ANCHOR_STACK_VAL,
            SAVED_ANCHOR_STACK_TEST,
            "Stacking blended with SimCLR No-TTA anchor",
        ),
    ),
    (
        "A12 + Class-Bias Correction (FINAL)",
        (
            SAVED_BIAS_VAL,
            SAVED_BIAS_TEST,
            "Validation-selected final proposed model",
        ),
    ),
    (
        "B1 DINOv2 ViT-S/14 3-seed Mean",
        (
            DINO_VAL,
            DINO_TEST,
            "Modern SSL baseline",
        ),
    ),
])

rows = []
METRIC_CACHE = {}

simclr_test_bacc = metrics_dict(
    Y_TEST,
    TEST_PROB["SimCLR_NoTTA"]
)["bacc"]

for stage_name, (
    val_p,
    test_p,
    description
) in STAGES.items():

    vm = metrics_dict(
        Y_VAL,
        val_p,
    )

    tm = metrics_dict(
        Y_TEST,
        test_p,
    )

    METRIC_CACHE[
        stage_name
    ] = {
        "validation": vm,
        "test": tm,
    }

    rows.append({
        "stage": stage_name,
        "description": description,
        "validation_bacc": vm["bacc"],
        "validation_macro_f1": vm["macro_f1"],
        "test_accuracy": tm["accuracy"],
        "test_bacc": tm["bacc"],
        "test_macro_f1": tm["macro_f1"],
        "test_macro_specificity": tm["macro_specificity"],
        "test_auc_macro_ovr": tm["auc_macro_ovr"],
        "delta_bacc_vs_simclr": tm["bacc"] - simclr_test_bacc,
    })

ABLATION_DF = pd.DataFrame(rows)

display(
    ABLATION_DF
)

ABLATION_DF.to_csv(
    TABLE_DIR / "journal_progressive_ablation_original_split.csv",
    index=False,
)

PERCENT_DF = ABLATION_DF.copy()

metric_cols = [
    "validation_bacc",
    "validation_macro_f1",
    "test_accuracy",
    "test_bacc",
    "test_macro_f1",
    "test_macro_specificity",
    "test_auc_macro_ovr",
    "delta_bacc_vs_simclr",
]

for c in metric_cols:
    PERCENT_DF[c] = 100.0 * PERCENT_DF[c]

PERCENT_DF.to_csv(
    TABLE_DIR / "journal_progressive_ablation_percent.csv",
    index=False,
)

display(PERCENT_DF)

                                  stage  \
0                           A1 Rotation   
1                      A2 SimCLR No-TTA   
2             A3 SimCLR + Rotation Mean   
3                A4 SimCLR Selected-TTA   
4                       A5 FusionConcat   
5                     A6 FusionGated V2   
6                     A7 FusionGated V3   
7                    A8 Residual Fusion   
8        A9 Optimized Weighted Ensemble   
9             A10 Raw Logistic Stacking   
10         A11 No-TTA Anchor + Stacking   
11  A12 + Class-Bias Correction (FINAL)   
12       B1 DINOv2 ViT-S/14 3-seed Mean   

                                   description  validation_bacc  \
0                            Single SSL branch         0.755004   
1                   Single SSL branch / anchor         0.787846   
2        Simple heterogeneous probability mean         0.787257   
3             Single SSL branch + selected TTA         0.793305   
4                               Feature fusion         0.78561

                                  stage  \
0                           A1 Rotation   
1                      A2 SimCLR No-TTA   
2             A3 SimCLR + Rotation Mean   
3                A4 SimCLR Selected-TTA   
4                       A5 FusionConcat   
5                     A6 FusionGated V2   
6                     A7 FusionGated V3   
7                    A8 Residual Fusion   
8        A9 Optimized Weighted Ensemble   
9             A10 Raw Logistic Stacking   
10         A11 No-TTA Anchor + Stacking   
11  A12 + Class-Bias Correction (FINAL)   
12       B1 DINOv2 ViT-S/14 3-seed Mean   

                                   description  validation_bacc  \
0                            Single SSL branch        75.500424   
1                   Single SSL branch / anchor        78.784646   
2        Simple heterogeneous probability mean        78.725689   
3             Single SSL branch + selected TTA        79.330510   
4                               Feature fusion        78.56165

In [12]:
# ============================================================
# CELL 12 — TRUE ENSEMBLE-PROGRESSION DELTAS
# ============================================================

ENSEMBLE_STAGES = [
    "A2 SimCLR No-TTA",
    "A9 Optimized Weighted Ensemble",
    "A10 Raw Logistic Stacking",
    "A11 No-TTA Anchor + Stacking",
    "A12 + Class-Bias Correction (FINAL)",
]

progression_rows = []

previous_bacc = None

for stage in ENSEMBLE_STAGES:
    vm = METRIC_CACHE[stage]["validation"]
    tm = METRIC_CACHE[stage]["test"]

    progression_rows.append({
        "stage": stage,
        "validation_bacc": vm["bacc"],
        "test_bacc": tm["bacc"],
        "test_macro_f1": tm["macro_f1"],
        "test_accuracy": tm["accuracy"],
        "incremental_test_bacc_gain": (
            np.nan
            if previous_bacc is None
            else tm["bacc"] - previous_bacc
        ),
        "gain_vs_simclr_test_bacc": (
            tm["bacc"]
            - METRIC_CACHE[
                "A2 SimCLR No-TTA"
            ]["test"]["bacc"]
        ),
    })

    previous_bacc = tm["bacc"]

ENSEMBLE_PROGRESSION_DF = pd.DataFrame(
    progression_rows
)

display(
    ENSEMBLE_PROGRESSION_DF
)

ENSEMBLE_PROGRESSION_DF.to_csv(
    TABLE_DIR / "ensemble_progression_deltas.csv",
    index=False,
)

                                 stage  validation_bacc  test_bacc  \
0                     A2 SimCLR No-TTA         0.787846   0.810729   
1       A9 Optimized Weighted Ensemble         0.817714   0.810289   
2            A10 Raw Logistic Stacking         0.815683   0.829945   
3         A11 No-TTA Anchor + Stacking         0.816158   0.839954   
4  A12 + Class-Bias Correction (FINAL)         0.820332   0.839549   

   test_macro_f1  test_accuracy  incremental_test_bacc_gain  \
0       0.814429       0.844515                         NaN   
1       0.812911       0.850829                   -0.000440   
2       0.774714       0.831097                    0.019657   
3       0.798800       0.837017                    0.010008   
4       0.797961       0.838990                   -0.000404   

   gain_vs_simclr_test_bacc  
0                  0.000000  
1                 -0.000440  
2                  0.019217  
3                  0.029225  
4                  0.028821  

In [13]:
# ============================================================
# CELL 13 — PER-CLASS RECALL ABLATION
# ============================================================

recall_rows = []

simclr_recalls = METRIC_CACHE[
    "A2 SimCLR No-TTA"
]["test"]["class_recalls"]

for stage_name in STAGES:
    recalls = METRIC_CACHE[
        stage_name
    ]["test"]["class_recalls"]

    row = {
        "stage": stage_name,
    }

    for class_id, class_name in enumerate(CLASS_NAMES):
        row[f"recall_{class_name}"] = float(
            recalls[class_id]
        )

        row[f"delta_vs_simclr_{class_name}"] = float(
            recalls[class_id]
            - simclr_recalls[class_id]
        )

    recall_rows.append(row)

PER_CLASS_RECALL_DF = pd.DataFrame(
    recall_rows
)

display(
    PER_CLASS_RECALL_DF
)

PER_CLASS_RECALL_DF.to_csv(
    TABLE_DIR / "per_class_recall_ablation.csv",
    index=False,
)

# Compact final vs SimCLR vs DINO table.
selected_stages = [
    "A2 SimCLR No-TTA",
    "A12 + Class-Bias Correction (FINAL)",
    "B1 DINOv2 ViT-S/14 3-seed Mean",
]

compact_rows = []

for stage_name in selected_stages:
    recalls = METRIC_CACHE[
        stage_name
    ]["test"]["class_recalls"]

    for class_id, class_name in enumerate(CLASS_NAMES):
        compact_rows.append({
            "model": stage_name,
            "class": class_name,
            "recall": recalls[class_id],
        })

COMPACT_CLASS_DF = pd.DataFrame(
    compact_rows
)

COMPACT_CLASS_DF.to_csv(
    TABLE_DIR / "final_simclr_dino_per_class_recall.csv",
    index=False,
)

                                  stage  recall_MEL  delta_vs_simclr_MEL  \
0                           A1 Rotation    0.665929            -0.095133   
1                      A2 SimCLR No-TTA    0.761062             0.000000   
2             A3 SimCLR + Rotation Mean    0.725664            -0.035398   
3                A4 SimCLR Selected-TTA    0.761062             0.000000   
4                       A5 FusionConcat    0.732301            -0.028761   
5                     A6 FusionGated V2    0.719027            -0.042035   
6                     A7 FusionGated V3    0.714602            -0.046460   
7                    A8 Residual Fusion    0.719027            -0.042035   
8        A9 Optimized Weighted Ensemble    0.741150            -0.019912   
9             A10 Raw Logistic Stacking    0.752212            -0.008850   
10         A11 No-TTA Anchor + Stacking    0.752212            -0.008850   
11  A12 + Class-Bias Correction (FINAL)    0.747788            -0.013274   
12       B1 

In [14]:
# ============================================================
# CELL 14 — JOURNAL FIGURES
# ============================================================

# Figure 1: Test BACC across methods
plot_df = PERCENT_DF.copy()

plt.figure(
    figsize=(10.5, 6.6)
)

y = np.arange(
    len(plot_df)
)

plt.barh(
    y,
    plot_df["test_bacc"]
)

plt.yticks(
    y,
    plot_df["stage"]
)

plt.xlabel(
    "Internal Test Balanced Accuracy (%)"
)

plt.title(
    "Progressive Ablation on the Original 80/10/10 Split"
)

plt.gca().invert_yaxis()
plt.grid(
    axis="x",
    alpha=0.2
)

for idx, value in enumerate(
    plot_df["test_bacc"]
):
    plt.text(
        value + 0.15,
        idx,
        f"{value:.2f}",
        va="center",
        fontsize=8,
    )

plt.tight_layout()

plt.savefig(
    FIG_DIR / "progressive_ablation_test_bacc.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

# Figure 2: True ensemble progression
ep = ENSEMBLE_PROGRESSION_DF.copy()

plt.figure(
    figsize=(9.2, 5.4)
)

plt.plot(
    np.arange(len(ep)),
    100 * ep["test_bacc"],
    marker="o",
)

plt.xticks(
    np.arange(len(ep)),
    [
        "SimCLR",
        "Weighted",
        "Raw stack",
        "Anchor blend",
        "+ Bias",
    ],
    rotation=20,
)

plt.ylabel(
    "Internal Test Balanced Accuracy (%)"
)

plt.title(
    "Ensemble Decision-Fusion Progression"
)

plt.grid(
    alpha=0.2
)

for idx, value in enumerate(
    100 * ep["test_bacc"]
):
    plt.text(
        idx,
        value + 0.25,
        f"{value:.2f}",
        ha="center",
        fontsize=9,
    )

plt.tight_layout()

plt.savefig(
    FIG_DIR / "ensemble_progression_bacc.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [15]:
# ============================================================
# CELL 15 — FINAL VALIDATION-SELECTION AUDIT
# ============================================================

final_validation = METRIC_CACHE[
    "A12 + Class-Bias Correction (FINAL)"
]["validation"]

final_test = METRIC_CACHE[
    "A12 + Class-Bias Correction (FINAL)"
]["test"]

no_bias_validation = METRIC_CACHE[
    "A11 No-TTA Anchor + Stacking"
]["validation"]

no_bias_test = METRIC_CACHE[
    "A11 No-TTA Anchor + Stacking"
]["test"]

selection_audit = {
    "final_method": EXPECTED_FINAL_METHOD,
    "selection_source": "validation only",
    "test_used_for_selection": False,
    "final_validation_bacc": final_validation["bacc"],
    "no_bias_validation_bacc": no_bias_validation["bacc"],
    "final_test_bacc": final_test["bacc"],
    "no_bias_test_bacc": no_bias_test["bacc"],
    "important_interpretation": (
        "The no-bias stacker is slightly higher on the held-out test set, "
        "but the class-bias version remains the frozen final method because "
        "it was selected using validation only."
    ),
    "dinov2_test_bacc": METRIC_CACHE[
        "B1 DINOv2 ViT-S/14 3-seed Mean"
    ]["test"]["bacc"],
    "proposed_minus_dino_test_bacc": (
        final_test["bacc"]
        - METRIC_CACHE[
            "B1 DINOv2 ViT-S/14 3-seed Mean"
        ]["test"]["bacc"]
    ),
    "proposed_minus_simclr_test_bacc": (
        final_test["bacc"]
        - METRIC_CACHE[
            "A2 SimCLR No-TTA"
        ]["test"]["bacc"]
    ),
}

with open(
    AUDIT_DIR / "J3_ablation_selection_audit.json",
    "w",
) as f:
    json.dump(
        selection_audit,
        f,
        indent=2,
        default=float,
    )

print(
    json.dumps(
        selection_audit,
        indent=2,
        default=float,
    )
)

{
  "final_method": "NoTTA_StackingBlend_ClassBias",
  "selection_source": "validation only",
  "test_used_for_selection": false,
  "final_validation_bacc": 0.8203316364511613,
  "no_bias_validation_bacc": 0.8161584317359797,
  "final_test_bacc": 0.8395491150909984,
  "no_bias_test_bacc": 0.8399535032220782,
  "important_interpretation": "The no-bias stacker is slightly higher on the held-out test set, but the class-bias version remains the frozen final method because it was selected using validation only.",
  "dinov2_test_bacc": 0.8043011301642334,
  "proposed_minus_dino_test_bacc": 0.03524798492676495,
  "proposed_minus_simclr_test_bacc": 0.028820579808856883
}


In [16]:
# ============================================================
# CELL 16 — COMPLETION MARKER + ZIP
# ============================================================

final_bacc = METRIC_CACHE[
    "A12 + Class-Bias Correction (FINAL)"
]["test"]["bacc"]

dino_bacc = METRIC_CACHE[
    "B1 DINOv2 ViT-S/14 3-seed Mean"
]["test"]["bacc"]

simclr_bacc = METRIC_CACHE[
    "A2 SimCLR No-TTA"
]["test"]["bacc"]

marker = {
    "stage": "J3_original_split_progressive_ablation",
    "status": "complete",
    "protocol": "original_stratified_80_10_10",
    "validation_n": len(Y_VAL),
    "test_n": len(Y_TEST),
    "final_method": EXPECTED_FINAL_METHOD,
    "final_test_bacc": float(final_bacc),
    "simclr_test_bacc": float(simclr_bacc),
    "dinov2_test_bacc": float(dino_bacc),
    "final_minus_simclr_bacc": float(
        final_bacc - simclr_bacc
    ),
    "final_minus_dinov2_bacc": float(
        final_bacc - dino_bacc
    ),
    "test_used_for_selection": False,
    "output_rel": "j3_original_split_ablation",
}

MARKER_PATH = WORKING / "J3_ABLATION_COMPLETE.json"

with open(MARKER_PATH, "w") as f:
    json.dump(
        marker,
        f,
        indent=2,
    )

shutil.copy2(
    MARKER_PATH,
    OUTPUT_DIR / "J3_ABLATION_COMPLETE.json",
)

ZIP_PATH = WORKING / "J3_ORIGINAL_SPLIT_ABLATION_RESULTS.zip"

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for p in sorted(
        OUTPUT_DIR.rglob("*")
    ):
        if p.is_file():
            zf.write(
                p,
                arcname=str(
                    p.relative_to(
                        OUTPUT_DIR.parent
                    )
                ),
            )

required = [
    TABLE_DIR / "journal_progressive_ablation_original_split.csv",
    TABLE_DIR / "ensemble_progression_deltas.csv",
    TABLE_DIR / "per_class_recall_ablation.csv",
    FIG_DIR / "progressive_ablation_test_bacc.png",
    FIG_DIR / "ensemble_progression_bacc.png",
    AUDIT_DIR / "J3_ablation_selection_audit.json",
    MARKER_PATH,
    ZIP_PATH,
]

missing = [
    str(p)
    for p in required
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing J3 outputs:\n"
        + "\n".join(missing)
    )

print("=" * 96)
print("J3 PROGRESSIVE ABLATION COMPLETE")
print("=" * 96)

print(
    "Final proposed BACC:",
    f"{100 * final_bacc:.4f}%"
)

print(
    "SimCLR BACC       :",
    f"{100 * simclr_bacc:.4f}%"
)

print(
    "DINOv2 BACC       :",
    f"{100 * dino_bacc:.4f}%"
)

print(
    "Final - SimCLR    :",
    f"{100 * (final_bacc - simclr_bacc):+.4f} pp"
)

print(
    "Final - DINOv2    :",
    f"{100 * (final_bacc - dino_bacc):+.4f} pp"
)

print(
    "ZIP:",
    ZIP_PATH
)

print()
print(
    "Save this Kaggle notebook Version WITH OUTPUTS "
    "and send the executed notebook back."
)

J3 PROGRESSIVE ABLATION COMPLETE
Final proposed BACC: 83.9549%
SimCLR BACC       : 81.0729%
DINOv2 BACC       : 80.4301%
Final - SimCLR    : +2.8821 pp
Final - DINOv2    : +3.5248 pp
ZIP: /kaggle/working/J3_ORIGINAL_SPLIT_ABLATION_RESULTS.zip

Save this Kaggle notebook Version WITH OUTPUTS and send the executed notebook back.
